# AME 5003 — Principles of Natural Language Processing
## Practical: Building a Text-Preprocessing Pipeline

**Manipal School of Information Sciences (MSIS)**  
**Manipal Academy of Higher Education (MAHE)**

### Topics
**Tokenization · Normalization · Stop-word removal · Stemming**

Real text rarely arrives as neat words separated by single spaces:

> `Amazing delivery!!! I can't believe it reached by 10:30 a.m. — cost ₹1,299.50 😊`

Before an NLP system can count words, search documents, or classify reviews, we must decide:

- What counts as a token?
- Which surface variations should be normalized?
- Which frequent words may be removed?
- Which related word forms should be reduced?

> **Preprocessing is not “clean everything.” It is “change only what the downstream task can afford to lose.”**

For every section use this pattern:

**Understand → Predict → Run → Compare → Break → Explain**

You may use documentation, web search, or AI tools, but you must be able to explain and test any code you use.

## 0. Learning outcomes

By the end of the practical you should be able to:

1. distinguish sentence tokenization from word tokenization;
2. explain why whitespace splitting is not a general tokenizer;
3. compare tokenization of contractions, hyphens, numbers, abbreviations, emails, URLs and emojis;
4. perform conservative whitespace and Unicode normalization;
5. explain why lowercasing and punctuation removal are task-dependent;
6. inspect a stop-word list and identify cases where removal changes meaning;
7. create a task-specific stop-word policy;
8. apply Porter stemming and diagnose useful, risky and surprising reductions;
9. connect stemming with vocabulary size and feature fragmentation;
10. build an end-to-end preprocessing pipeline for a realistic text-analytics application.

### Working sequence for this practical

**Raw text → Tokenization → Normalization → Stop-word decision → Stemming → Processed representation**

This is a useful sequence for today's exercises, **not a universal recipe for every NLP application**.

## 0.1 Setup and common examples

The examples come from everyday applications: shopping reviews, hotel reviews, customer support and social-media-like text.

Run this cell first.

### Libraries used in this practical

This notebook uses a few standard Python and NLTK tools for text preprocessing.

- **`re`**  
  Python's regular-expression library. We use it mainly for simple text cleaning tasks such as handling repeated whitespace and pattern-based text operations.

- **`unicodedata`**  
  Provides Unicode utilities. It is useful for normalizing text that may look identical to us but be represented differently internally.

- **`Counter` from `collections`**  
  Counts how often items occur. Later we use it to count processed words and identify frequent terms.

- **`nltk`**  
  The Natural Language Toolkit. It provides commonly used NLP resources and preprocessing tools.

The following NLTK resources are downloaded:

- **`punkt` / `punkt_tab`** — resources used by NLTK tokenizers for sentence and word boundary detection.
- **`stopwords`** — standard lists of frequently occurring words for several languages.

We then import the tools needed for this practical:

- **`word_tokenize`** — tokenizes text into word-level tokens.
- **`sent_tokenize`** — separates text into sentences.
- **`TweetTokenizer`** — a tokenizer designed to handle informal and social-media-style text.
- **`stopwords`** — provides the NLTK stop-word lists.
- **`PorterStemmer`** — implements the Porter stemming algorithm.

Finally, we create three reusable objects:

- **`tweet_tokenizer`** — an instance of `TweetTokenizer`.
- **`stemmer`** — an instance of `PorterStemmer`.
- **`english_stopwords`** — the English stop-word list converted to a Python `set` for efficient lookup.

You do not need to memorize these commands now.  
We will use each of these tools step by step in the sections that follow.

In [1]:
import re
import unicodedata
from collections import Counter

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

tweet_tokenizer = TweetTokenizer()
stemmer = PorterStemmer()
english_stopwords = set(stopwords.words("english"))

texts = [
    "Amazing delivery!!! I can't believe it reached by 10:30 a.m. — cost ₹1,299.50 😊",
    "The hotel was clean, but the room wasn't quiet at night.",
    "DO NOT cancel my order. I still want the blue headphones.",
    "Email me at priya.nair@example.com or visit https://example.com/help.",
    "The battery-life is good; however, fast-charging isn't working.",
    "I loved the café near the station. The cafe\u0301 next door was closed.",
    "No refund has been received yet.",
    "Running shoes are comfortable for runners who run every morning.",
    "Studies show that studying regularly improves study habits.",
    "The UNIVERSAL adapter works worldwide, but the university shop does not stock it.",
    "This phone is not bad at all — actually, it's quite good!"
]

for i, t in enumerate(texts, 1):
    print(f"{i:02d}: {repr(t)}")

01: "Amazing delivery!!! I can't believe it reached by 10:30 a.m. — cost ₹1,299.50 😊"
02: "The hotel was clean, but the room wasn't quiet at night."
03: 'DO NOT cancel my order. I still want the blue headphones.'
04: 'Email me at priya.nair@example.com or visit https://example.com/help.'
05: "The battery-life is good; however, fast-charging isn't working."
06: 'I loved the café near the station. The café next door was closed.'
07: 'No refund has been received yet.'
08: 'Running shoes are comfortable for runners who run every morning.'
09: 'Studies show that studying regularly improves study habits.'
10: 'The UNIVERSAL adapter works worldwide, but the university shop does not stock it.'
11: "This phone is not bad at all — actually, it's quite good!"


### Why `repr()` instead of `str()` (and other relevant points)

Both `repr()` and `str()` convert an object into a string representation, but they serve different purposes:

*   **`str()` (for `print()`):** Aims to produce a human-readable representation of an object. It's what you typically see when you `print()` a variable. It often hides details that are not important for general display, such as internal quotes or escape sequences, and may interpret special characters.
*   **`repr()` (for developers/debugging):** Aims to produce an unambiguous, 'developer-friendly' representation of an object. The goal is often that `eval(repr(obj))` would return an object equivalent to `obj`. It explicitly shows string delimiters (quotes) and uses escape sequences for non-printable characters (like `\n` for newline or `\u` for Unicode characters), making the exact content of the string clear.

**Why `repr()` was used here:**

1.  **Debugging and Inspection:** In text preprocessing, we often deal with raw text that can contain hidden characters (like newlines, tabs, non-breaking spaces) or subtle Unicode differences that `print()` might obscure. `repr()` makes these explicit, which is crucial for understanding exactly what's in the string and how it might behave with different tokenizers or normalizers.
2.  **Unambiguous Representation:** It ensures that you see the exact content of each string, including its boundaries (with quotes) and any escape sequences. For instance, if a string accidentally has a leading space, `repr()` will show it clearly, whereas `str()` (or `print()`) might make it harder to spot.
3.  **Revealing Special Characters:** When we print `example_text_with_special_chars`, the newline `\n` is interpreted, creating a line break. When we `repr()` it, we see `\n` as part of the string, confirming its presence explicitly. Similarly, Unicode combining characters (like the accent in `café` in `texts[5]`) are shown with their escape sequences if they are not standard printable characters, highlighting potential normalization issues.
4.  **Preserving Raw Data View:** In the context of NLP preprocessing, we are often working with raw or semi-raw text where every character matters. `repr()` gives us a more faithful representation of the underlying data for analytical purposes, rather than a prettified, human-readable version that `str()` provides.

# Part 1 — Tokenization: what should count as one unit?

## 1.1 Theory first

A **token** is a unit passed to later NLP stages.

For:

`The parcel arrived today.`

a reasonable tokenization is:

`The | parcel | arrived | today | .`

But real text contains difficult boundaries:

- `can't`
- `battery-life`
- `₹1,299.50`
- `10:30`
- `a.m.`
- `priya.nair@example.com`
- `https://example.com/help`
- `😊`
- `#GreatBuy`

There is often no single universally correct answer.

> **The useful question is: what downstream task will consume these tokens?**

For sentiment, `not` and emojis may matter.  
For information extraction, an email may need to remain intact.  
For search, `battery-life` and `battery life` may need consistent treatment.

### Sentence tokenization vs word tokenization

**Sentence tokenization** asks: where does one sentence end and another begin?  
**Word/token tokenization** asks: what units should occur inside each sentence?

## 1.2 First baseline: whitespace splitting

Python's `.split()` splits only at whitespace.

Before running the cell, predict what happens to:

- `can't`
- `10:30`
- `a.m.`
- `₹1,299.50`
- `delivery!!!`
- `😊`

In [2]:
example = texts[0]

print("TEXT:")
print(example)

print("\nWhitespace split:")
print(example.split())

TEXT:
Amazing delivery!!! I can't believe it reached by 10:30 a.m. — cost ₹1,299.50 😊

Whitespace split:
['Amazing', 'delivery!!!', 'I', "can't", 'believe', 'it', 'reached', 'by', '10:30', 'a.m.', '—', 'cost', '₹1,299.50', '😊']


### What should you notice?

`.split()` is useful as a **baseline**, but it knows nothing about punctuation or linguistic structure.

For example, `delivery!!!` remains one item.

That is not automatically wrong, it simply means the only boundary rule was whitespace.

## 1.3 Compare three tokenizers

We will compare:

1. `.split()`
2. NLTK `word_tokenize()`
3. NLTK `TweetTokenizer()`

### Activity 1.1

Test:

- `I can't believe this!!!`
- `The battery-life is excellent.`
- `The price is ₹1,299.50.`
- `Email priya.nair@example.com or visit https://example.com/help.`
- `Loved it!!! 😊 #GreatBuy`

Before running, think about what all **you** would prefer each tokenizer to preserve.

In [3]:
token_tests = [
    "I can't believe this!!!",
    "The battery-life is excellent.",
    "The price is ₹1,299.50.",
    "Email priya.nair@example.com or visit https://example.com/help.",
    "Loved it!!! 😊 #GreatBuy"
]

for text in token_tests:
    print("\nTEXT:", text)
    print("split():", text.split())

    # TODO 1: printword_tokenize(text)
    print("word_tokenize: ", word_tokenize(text))
    # TODO 2: print tweet_tokenizer.tokenize(text)
    print('tweet tokenze:', tweet_tokenizer.tokenize(text))


TEXT: I can't believe this!!!
split(): ['I', "can't", 'believe', 'this!!!']
word_tokenize:  ['I', 'ca', "n't", 'believe', 'this', '!', '!', '!']
tweet tokenze: ['I', "can't", 'believe', 'this', '!', '!', '!']

TEXT: The battery-life is excellent.
split(): ['The', 'battery-life', 'is', 'excellent.']
word_tokenize:  ['The', 'battery-life', 'is', 'excellent', '.']
tweet tokenze: ['The', 'battery-life', 'is', 'excellent', '.']

TEXT: The price is ₹1,299.50.
split(): ['The', 'price', 'is', '₹1,299.50.']
word_tokenize:  ['The', 'price', 'is', '₹1,299.50', '.']
tweet tokenze: ['The', 'price', 'is', '₹', '1,299', '.', '50', '.']

TEXT: Email priya.nair@example.com or visit https://example.com/help.
split(): ['Email', 'priya.nair@example.com', 'or', 'visit', 'https://example.com/help.']
word_tokenize:  ['Email', 'priya.nair', '@', 'example.com', 'or', 'visit', 'https', ':', '//example.com/help', '.']
tweet tokenze: ['Email', 'priya.nair@example.com', 'or', 'visit', 'https://example.com/help', 

## 1.4 Confusing token boundaries

The same surface form may be treated differently depending on the task.

| Expression | Treatment A | Treatment B |
|---|---|---|
| `can't` | keep as one token | split into contraction parts |
| `battery-life` | keep compound | split around hyphen |
| `₹1,299.50` | one monetary token | currency + numeric value |
| `New York` | two word tokens | one entity span later |
| `#GreatBuy` | preserve hashtag | normalize to `greatbuy` |
| `😊` | preserve sentiment signal | discard for some retrieval tasks |

### Activity 1.2 — Make the decision before coding

For each expression below, choose a preferred treatment for:

**A. product-review sentiment analysis**  
**B. product search**

Expressions:

`not good`, `battery-life`, `₹1,299.50`, `wireless-earbuds`, `#GreatBuy`, `😊`

Discuss one case where your decision differs between A and B.

## 1.5 Sentence tokenization has exceptions too

A naive rule such as “split at every period” fails for:

`Dr. Rao arrived at 10.30 a.m. He checked into the hotel.`

Periods occur inside `Dr.`, `10.30`, and `a.m.`.

### Activity 1.3

1. Predict the number of sentences.
2. Use `sent_tokenize`.
3. Add one difficult example containing an abbreviation, decimal, or time.
4. Check whether the tokenizer behaves as expected.

In [4]:
sentence_text = "Dr. Rao arrived at 10.30 a.m. He checked into the hotel."

# TODO: use sent_tokenize and print each sentence separately.
sentences = sent_tokenize(sentence_text)
for i, s in enumerate(sentences,1):
    print(f"Sentence {i}: {s}")

# Add your own difficult case.
my_sentence_case = "Mr. Smith bought 1,000 shares of Acme Corp. at $10.50 each on Jan. 15th, 2020."
sample=sent_tokenize(my_sentence_case)
print(sample)
# for i,t in enumerate(sample,1):
#     print(f"Sentence {i}: {t}")

Sentence 1: Dr. Rao arrived at 10.30 a.m.
Sentence 2: He checked into the hotel.
['Mr. Smith bought 1,000 shares of Acme Corp. at $10.50 each on Jan. 15th, 2020.']


---

## 🛑 CHECKPOINT 1 — Tokenization

You should now be able to explain:

- Why is `.split()` only a baseline?
- Why can `can't` have more than one defensible tokenization?
- Why may email addresses, URLs and money need special handling?
- Why is `.` not enough for sentence segmentation?
- Give one boundary decision that depends on the downstream task.

---

# Part 2 — Normalization: reduce unnecessary surface variation

## 2.1 Theory first

Normalization makes selected variants more consistent.

Example:

```text
"   Delivery   was delayed.\n\nSupport was helpful.   "
```

may reasonably become:

```text
"Delivery was delayed. Support was helpful."
```

Repeated spacing is usually layout noise.

But other operations are riskier.

Compare:

`US` vs `us`

Lowercasing destroys the distinction.

Compare:

`GOOD!!!` vs `good`

For sentiment or urgency, capitalization and punctuation may carry evidence.

### Lower-risk normalization

- Unicode normalization
- trimming leading/trailing spaces
- collapsing repeated whitespace

### Task-dependent normalization

- lowercasing
- deleting punctuation
- removing emojis
- deleting URLs/emails
- expanding contractions
- standardizing spelling

## 2.2 Meaningful whitespace normalization

Use `repr()` so that spaces and newline characters are visible.

In [5]:
messy = "   Delivery    was delayed.\n\nCustomer   support was helpful.   "

normalized_ws = re.sub(r"\s+", " ", messy).strip()

print("BEFORE:", repr(messy))
print("AFTER :", repr(normalized_ws))

BEFORE: '   Delivery    was delayed.\n\nCustomer   support was helpful.   '
AFTER : 'Delivery was delayed. Customer support was helpful.'


## 2.3 Unicode normalization — visually identical can be computationally different

These may look the same:

`café`

`café`

But one can be stored with a precomposed `é`, and the other as `e` + a combining accent.

Run the example.

In [6]:
u1 = "café"
u2 = "cafe\u0301"

print("u1:", u1)
print("u2:", u2)
print("Equal before normalization?", u1 == u2)

n1 = unicodedata.normalize("NFC", u1)
n2 = unicodedata.normalize("NFC", u2)

print("Equal after NFC?", n1 == n2)

u1: café
u2: café
Equal before normalization? False
Equal after NFC? True


### A short guide to Unicode normalization

The same visible text can sometimes be stored using different Unicode character sequences.

For example, **é** may be stored as:

- one character: `é`
- two characters: `e` + combining accent

Unicode normalization converts such representations into a consistent form.

| Form | Meaning | What it does |
|---|---|---|
| **NFC** | Canonical Composition | Combines characters where possible. Usually a good default for normal text. |
| **NFD** | Canonical Decomposition | Splits characters into their basic components. |
| **NFKC** | Compatibility Composition | Like NFC, but also converts some stylistic/compatibility variants to simpler forms. |
| **NFKD** | Compatibility Decomposition | Like NFD, with additional compatibility simplification. |

#### Example

`café`

may internally contain one `é`, while

`café`

may contain `e` + a separate accent.

They look identical to us but may compare as different strings.

**NFC** makes these canonically equivalent representations consistent.

> For ordinary NLP text, **NFC is a safe and common starting point**.  
> NFKC performs stronger normalization and should be used only when those additional conversions are desirable.

Test the code snippet below for comparison.

In [7]:
# Comparing different forms of Unicode Normalization

import unicodedata

text = "cafe\u0301 ① ²"

print("Original:", repr(text))
print()

for form in ["NFC", "NFD", "NFKC", "NFKD"]:
    normalized = unicodedata.normalize(form, text)

    print(f"{form}:")
    print(" Text       :", normalized)
    print(" repr()     :", repr(normalized))
    print(" Length     :", len(normalized))
    print(" Code points:", [hex(ord(c)) for c in normalized])
    print()

Original: 'café ① ²'

NFC:
 Text       : café ① ²
 repr()     : 'café ① ²'
 Length     : 8
 Code points: ['0x63', '0x61', '0x66', '0xe9', '0x20', '0x2460', '0x20', '0xb2']

NFD:
 Text       : café ① ²
 repr()     : 'café ① ²'
 Length     : 9
 Code points: ['0x63', '0x61', '0x66', '0x65', '0x301', '0x20', '0x2460', '0x20', '0xb2']

NFKC:
 Text       : café 1 2
 repr()     : 'café 1 2'
 Length     : 8
 Code points: ['0x63', '0x61', '0x66', '0xe9', '0x20', '0x31', '0x20', '0x32']

NFKD:
 Text       : café 1 2
 repr()     : 'café 1 2'
 Length     : 9
 Code points: ['0x63', '0x61', '0x66', '0x65', '0x301', '0x20', '0x31', '0x20', '0x32']



## 2.4 Lowercasing: useful but use carefully

Examine:

1. `The US model is sold in India.`
2. `Please contact us tomorrow.`
3. `GOOD support!!!`
4. `Do NOT cancel my booking.`

### Activity 2.1

For each sentence, ask what is lost after lowercasing.

Would that loss matter for:

- topic analysis?
- sentiment analysis?
- named entity recognition?
- urgency detection?

### Important distinction

Changing `NOT` to `not` preserves negation but loses **emphasis**.  
Changing `US` to `us` can change the apparent meaning completely.

## 2.5 Build a conservative normalizer

For:

```text
"   DO NOT cancel my order!!!\nOrder value: ₹1,299.50   "
```

assume the downstream task is **customer-support triage**.

Your function must:

1. Unicode-normalize the text;
2. collapse repeated whitespace;
3. trim the ends.

Do **not** automatically remove case, punctuation or the monetary form unless you can justify it.

In [8]:
messy_support = "   DO NOT cancel my order!!!\nOrder value: ₹1,299.50   "

def conservative_normalize(text):
    # TODO 1: Unicode normalization
    text = unicodedata.normalize("NFC", text)
    print("NFC:", repr(text))
    # TODO 2: collapse repeated whitespace
    text= re.sub(r"\s+"," ",text)
    print("Collapsed whitespace:", repr(text))
    # TODO 3: strip leading/trailing whitespace
    text=text.strip()
    print("strip leading/trailing whitespace", repr(text))
    return text

print("RAW :", repr(messy_support))
print("NORM:", repr(conservative_normalize(messy_support)))

RAW : '   DO NOT cancel my order!!!\nOrder value: ₹1,299.50   '
NFC: '   DO NOT cancel my order!!!\nOrder value: ₹1,299.50   '
Collapsed whitespace: ' DO NOT cancel my order!!! Order value: ₹1,299.50 '
strip leading/trailing whitespace 'DO NOT cancel my order!!! Order value: ₹1,299.50'
NORM: 'DO NOT cancel my order!!! Order value: ₹1,299.50'


# Part 3 — Stop-word removal: frequent does not mean useless

## 3.1 Theory first

Stop words are frequent words that are sometimes removed before count-based representations.

Examples include:

`the, is, at, of, a, an`

But consider:

`The hotel is not clean.`

If `the`, `is`, and `not` disappear, the representation may become:

`hotel clean`

The meaning has changed.

> **A stop word is not inherently meaningless. Its usefulness depends on the task.**

## 3.2 Inspect the actual stop-word list

Do not assume which words are present. Check the NLTK list.

Then process:

`The hotel is not clean.`

In [9]:
print("'not' in stop words?", "not" in english_stopwords)
print("'no'  in stop words?", "no" in english_stopwords)
print("'nor' in stop words?", "nor" in english_stopwords)
print(type(english_stopwords))
example = "The hotel is not clean."
tokens = [t.lower() for t in word_tokenize(example) if t.isalpha()]
filtered = [t for t in tokens if t not in english_stopwords]

print("TOKENS  :", tokens)
print("FILTERED:", filtered)

'not' in stop words? True
'no'  in stop words? True
'nor' in stop words? True
<class 'set'>
TOKENS  : ['the', 'hotel', 'is', 'not', 'clean']
FILTERED: ['hotel', 'clean']


## 3.3 Confusing words: negation and contrast

For sentiment, complaints and reviews, pay attention to:

- `not`
- `no`
- `never`
- `without`
- `but`
- `however`

Examples:

- `The room was small but clean.`
- `The food was good, however service was slow.`
- `No refund was received.`
- `I would never buy this again.`

`but` and `however` are especially interesting because they signal **contrast**.

### Activity 3.1

Apply the default stop-word list to:

1. `This product is not good.`
2. `No refund has been received.`
3. `The hotel was small but clean.`
4. `The food was good, however service was slow.`
5. `I would never buy this again.`

Identify outputs that become misleading or less useful.

In [10]:
stopword_tests = [
    "This product is not good.",
    "No refund has been received.",
    "The hotel was small but clean.",
    "The food was good, however service was slow.",
    "I would never buy this again."
]

for text in stopword_tests:
    # TODO:
    # 1. tokenize
    token=[t.lower() for t in word_tokenize(text) if t.isalpha()]
    # 2. lowercase alphabetic tokens
    # 3. remove english_stopwords         
    filtered = [t for t in token if t not in english_stopwords]
    # 4. print BEFORE and AFTER

    print(f"BEFORE: {token}")
    print(f"AFTER:  {filtered}")

BEFORE: ['this', 'product', 'is', 'not', 'good']
AFTER:  ['product', 'good']
BEFORE: ['no', 'refund', 'has', 'been', 'received']
AFTER:  ['refund', 'received']
BEFORE: ['the', 'hotel', 'was', 'small', 'but', 'clean']
AFTER:  ['hotel', 'small', 'clean']
BEFORE: ['the', 'food', 'was', 'good', 'however', 'service', 'was', 'slow']
AFTER:  ['food', 'good', 'however', 'service', 'slow']
BEFORE: ['i', 'would', 'never', 'buy', 'this', 'again']
AFTER:  ['would', 'never', 'buy']


## 3.4 Build a task-specific stop-word policy

Assume the task is:

> **sentiment / complaint analysis**

Preserve at least:

`not`, `no`, `nor`

Consider whether you also want:

`never`, `without`, `but`, `however`

### Activity 3.2

Create a custom stop-word set and compare the output with the default list.

### Remember

For POS tagging, NER, parsing or question answering, stop-word removal may be inappropriate altogether.

In [11]:
# TODO: start with the standard list, but preserve task-important words.
preserved_words = {"not", "no", "nor","never", "without", "but", "however"}
custom_stopwords = set(preserved_words)

def remove_stopwords_for_sentiment(text):
    # TODO:
    token  =[t.lower() for t in word_tokenize(text) if t.isalpha()]
    return [t for t in token if t not in custom_stopwords]

for text in stopword_tests:
    print("\nTEXT:", text)
    print("CUSTOM:", remove_stopwords_for_sentiment(text))


TEXT: This product is not good.
CUSTOM: ['this', 'product', 'is', 'good']

TEXT: No refund has been received.
CUSTOM: ['refund', 'has', 'been', 'received']

TEXT: The hotel was small but clean.
CUSTOM: ['the', 'hotel', 'was', 'small', 'clean']

TEXT: The food was good, however service was slow.
CUSTOM: ['the', 'food', 'was', 'good', 'service', 'was', 'slow']

TEXT: I would never buy this again.
CUSTOM: ['i', 'would', 'buy', 'this', 'again']


---

## 🛑 CHECKPOINT 2 — Normalization and stop words

You should now be able to answer:

- Why is whitespace normalization usually safer than lowercasing?
- Why can `US` / `us` be a problem?
- Why can stop-word removal reverse sentiment?
- Why can `but` and `however` matter?
- Why should stop-word removal depend on the task?

---

# Part 4 — Stemming: reducing related word forms

## 4.1 Why stemming exists

Suppose the corpus contains:

`connect, connected, connecting, connects`

A Bag-of-Words representation may initially treat these as separate features.

Evidence for one concept becomes **fragmented** across several vocabulary dimensions.

Stemming applies heuristic rules to reduce related surface forms.

For example:

`studies → studi`  
`studying → studi`

A stem does **not** need to be a dictionary word.

The goal is vocabulary reduction, not perfect linguistic analysis.

## 4.2 First experiment

Predict which words will reduce to the same stem, then run:

- `connect`
- `connected`
- `connecting`
- `studies`
- `studying`
- `relational`
- `universities`

In [12]:
stem_demo_words = [
    "connect", "connected", "connecting",
    "studies", "studying",
    "relational", "universities"
]

for word in stem_demo_words:
    print(f"{word:14} -> {stemmer.stem(word)}")

connect        -> connect
connected      -> connect
connecting     -> connect
studies        -> studi
studying       -> studi
relational     -> relat
universities   -> univers


## 4.3 Stemming has failure modes

### Over-stemming
Different words are reduced too aggressively and become difficult to distinguish.

### Under-stemming
Related words remain separate.

Test these confusing words rather than memorizing what a stemmer “should” do:

- `university`
- `universal`
- `policy`
- `police`
- `analysis`
- `analyst`
- `organization`
- `organize`
- `organism`

### Activity 4.1

Print the actual Porter stems.

Classify each result as:

- useful;
- harmless;
- risky;
- surprising.

Explain at least one classification.

In [13]:
confusing_words = [
    "university", "universal",
    "policy", "police",
    "analysis", "analyst",
    "organization", "organize", "organism"
]

# TODO: print word -> Porter stem for every item.
print("\nConfusing words and their stems:")
for word in confusing_words:
    print(f"{word:14} -> {PorterStemmer().stem(word)}") 


Confusing words and their stems:
university     -> univers
universal      -> univers
policy         -> polici
police         -> polic
analysis       -> analysi
analyst        -> analyst
organization   -> organ
organize       -> organ
organism       -> organ


## 4.4 Measure vocabulary reduction

Use:

`run, runs, running, runner, connect, connected, connecting, connection, study, studies, studying, student`

Calculate:

1. number of unique original words;
2. number of unique stems;
3. percentage reduction.

Then inspect whether the reduction was actually sensible.

This connects stemming to:

- vocabulary size;
- sparsity;
- feature fragmentation.

In [14]:
vocab_words = [
    "run", "runs", "running", "runner",
    "connect", "connected", "connecting", "connection",
    "study", "studies", "studying", "student"
]

stem_map = {}
original_size = None
stem_size = None
reduction_pct = None

# TODO:
# 1. fill stem_map
stem_map = {w: stemmer.stem(w) for w in vocab_words}
# 2. calculate original_size
original_size = len(vocab_words)
# 3. calculate stem_size 
stem_size = len(set(stem_map.values()))
# 4. calculate percentage reduction
reduction_pct = ((original_size - stem_size) / original_size) * 100
print("Stem map:", stem_map)
print("Original size:", original_size)
print("Stem size:", stem_size)
print("Reduction %:", reduction_pct)

Stem map: {'run': 'run', 'runs': 'run', 'running': 'run', 'runner': 'runner', 'connect': 'connect', 'connected': 'connect', 'connecting': 'connect', 'connection': 'connect', 'study': 'studi', 'studies': 'studi', 'studying': 'studi', 'student': 'student'}
Original size: 12
Stem size: 5
Reduction %: 58.333333333333336


In [15]:
vocab_words = [
    "run", "runs", "running", "runner",
    "connect", "connected", "connecting", "connection",
    "study", "studies", "studying", "student"
]

stem_map = {w: stemmer.stem(w) for w in vocab_words}
original_size = len(set(vocab_words))
stem_size = len(set(stem_map.values()))
reduction_pct = 100 * (original_size - stem_size) / original_size

for w, s in stem_map.items():
    print(f"{w:12} -> {s}")

print("\nOriginal size:", original_size)
print("Stem size:", stem_size)
print("Reduction %:", round(reduction_pct, 2))

run          -> run
runs         -> run
running      -> run
runner       -> runner
connect      -> connect
connected    -> connect
connecting   -> connect
connection   -> connect
study        -> studi
studies      -> studi
studying     -> studi
student      -> student

Original size: 12
Stem size: 5
Reduction %: 58.33


# Part 5 — End-to-end application: product-review topic analysis

A company receives thousands of short reviews and wants a simple dashboard of recurring themes such as:

- battery
- delivery
- refund
- comfort
- quality
- service

For this **topic-frequency** objective, a defensible pipeline may:

- normalize Unicode;
- normalize whitespace;
- lowercase;
- tokenize;
- discard punctuation;
- remove many stop words;
- stem related word forms.

This would **not automatically be the right pipeline for sentiment analysis**.

## 5.1 Read the dataset first

Before processing, predict which themes should be frequent.

In [16]:
reviews = [
    "The battery lasts all day and charging is fast.",
    "Battery life is excellent, but the charger feels cheap.",
    "Delivery was late, although the package arrived safely.",
    "The delivery driver was helpful and the package was clean.",
    "I requested a refund because the headphones stopped working.",
    "Refund processing was slow but customer service was helpful.",
    "These running shoes are comfortable for long runs.",
    "The shoes felt uncomfortable after running for two hours.",
    "Excellent sound quality and very comfortable ear cushions.",
    "The build quality is good, but battery performance is average."
]

for i, r in enumerate(reviews, 1):
    print(f"{i:02d}. {r}")

01. The battery lasts all day and charging is fast.
02. Battery life is excellent, but the charger feels cheap.
03. Delivery was late, although the package arrived safely.
04. The delivery driver was helpful and the package was clean.
05. I requested a refund because the headphones stopped working.
06. Refund processing was slow but customer service was helpful.
07. These running shoes are comfortable for long runs.
08. The shoes felt uncomfortable after running for two hours.
09. Excellent sound quality and very comfortable ear cushions.
10. The build quality is good, but battery performance is average.


## 5.2 Design before coding

For the topic-frequency objective, decide:

| Step | Use? | Why? |
|---|---|---|
| Unicode normalization |  |  |
| whitespace normalization |  |  |
| lowercasing |  |  |
| tokenization |  |  |
| punctuation removal |  |  |
| stop-word removal |  |  |
| preserve negation |  |  |
| stemming |  |  |

### Question

Even for topic analysis, should `not` always be removed?

Different groups may choose differently. The important part is the reasoning.

In [17]:
topic_stopwords={"not", "no", "nor","never", "without", "but", "however"}
def preprocess_for_topics(text):
    """
    Convert one review into processed tokens suitable
    for simple topic-frequency analysis.
    """

    # TODO 1: Unicode normalization
    text=unicodedata.normalize("NFC",text)
    # TODO 2: whitespace normalization
    text=re.sub(r"\s+"," ", text).strip()
    # TODO 3: lowercasing if justified
    # TODO 4: tokeniztation
    token=[t.lower() for t in word_tokenize(text) if t.isalpha()]
    # TODO 5: retain useful token types
    token=[t for t in token if t not in topic_stopwords]
    # TODO 6: task-aware stop-word removal
    # TODO 7: stemming
    token = [stemmer.stem(t) for t in token]

    return token


print("RAW:")
print(reviews[0])

print("\nPROCESSED:")
print(preprocess_for_topics(reviews[0]))

RAW:
The battery lasts all day and charging is fast.

PROCESSED:
['the', 'batteri', 'last', 'all', 'day', 'and', 'charg', 'is', 'fast']


## 5.3 Apply the pipeline to the collection

1. preprocess every review;
2. combine all processed tokens;
3. calculate frequencies;
4. print the 15 most frequent processed terms.

Then ask:

- Do the terms correspond to recognizable themes?
- Did stemming combine useful variants?
- Did strange stems appear?
- Did an important word disappear because of your stop-word policy?

In [18]:
all_tokens = []

# TODO: preprocess each review and extend all_tokens.
for review in reviews:
    all_tokens.extend(preprocess_for_topics(review))
# TODO: use Counter to calculate frequencies.
topic_counts = Counter(all_tokens)
print("Top 15 most common terms:")
for term , count in topic_counts.most_common(15):
    print(f"{term:13}: {count}")


Top 15 most common terms:
the          : 8
wa           : 5
is           : 4
batteri      : 3
and          : 3
run          : 3
excel        : 2
deliveri     : 2
packag       : 2
help         : 2
refund       : 2
shoe         : 2
comfort      : 2
for          : 2
qualiti      : 2


## 5.4 Same text, different task

Consider:

`The battery is not bad, but charging is slow.`

A topic pipeline may focus on something like:

`batteri, bad, charg, slow`

But for sentiment analysis, removing:

`not` and `but`

could seriously damage interpretation.

> **A preprocessing pipeline is part of the model design.**

The same raw text may need different preprocessing for topic analysis, sentiment, search, POS tagging, NER or question answering.

# Part 6 — Reflection / mini-viva

1. What is the difference between sentence tokenization and word tokenization?
2. Give one reason `.split()` is not enough.
3. Why can `can't` be tokenized in different ways?
4. Why is whitespace normalization relatively low-risk?
5. Why is Unicode normalization useful?
6. Give one case where lowercasing removes useful information.
7. Why can stop-word removal change sentiment?
8. Why might `but` or `however` matter?
9. What is feature fragmentation?
10. Why can a stem be a non-dictionary word?
11. Give one surprising Porter stem you observed.
12. What is one limitation of your final pipeline?
13. Which preprocessing decision would you change first if the task became sentiment analysis?

### Submission checklist

- [ ] Three tokenizers compared.
- [ ] Sentence tokenization tested.
- [ ] At least one ambiguous token boundary explained.
- [ ] Unicode + whitespace normalization implemented.
- [ ] Default stop words tested on negation/contrast.
- [ ] Task-aware stop-word list created.
- [ ] Porter stemming tested on ordinary and confusing words.
- [ ] Vocabulary reduction measured.
- [ ] End-to-end topic pipeline implemented.
- [ ] Term-frequency output generated.
- [ ] One limitation documented.